#  ⭐ `fat_reacoes`


In [1]:
%run ../../config/bootstrap.py

import pandas as pd
import numpy as np
from pathlib import Path
from utils import get_project_root, normalize_date_column, normalize_rows, apply_mappings
project_root = get_project_root() 

In [2]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
path = Path(project_root) / "data/01_bronze/anvisa/Reacoes/Reacoes.parquet"
bronze = pd.read_parquet(path) 
bronze.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'REACAO_EVTO_ADVERSO_MEDDRA_LLT', 'PT',
       'HLT', 'HLGT', 'SOC', 'DATA_INICIO_HORA', 'DATA_FINAL_HORA', 'DURACAO',
       'GRAVE', 'GRAVIDADE', 'DESFECHO', 'ATUALIZADO'],
      dtype='object')

In [4]:
bronze.head(2)

,IDENTIFICACAO_NOTIFICACAO,REACAO_EVTO_ADVERSO_MEDDRA_LLT,PT,HLT,HLGT,SOC,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO
0,BR-ANVISA-300000004,Coceira,Prurido,Prurido NCO,Quadros clínicos epidérmicos e dérmicos,Distúrbios dos tecidos cutâneos e subcutâneos,None,None,3 dia,Não,None,Recuperado/Resolvido,2026-01-15
1,BR-ANVISA-300000005,Edema periorbital,Edema periorbital,Distúrbios oculares NCO,Transtornos oculares NCO,Distúrbios oculares,20181122,20181122,None,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2026-01-15


### GRAVE

In [5]:
bronze.GRAVE.value_counts()

GRAVE
Não    405156
Sim    285554
Name: count, dtype: int64

### DESFECHO

In [6]:
bronze.DESFECHO.value_counts()

DESFECHO
Recuperado/Resolvido                         303064
Desconhecido                                 240842
Não Recuperado/Não Resolvido/Em andamento     90692
Em recuperação/Resolvendo                     72126
Fatal/Óbito                                   19617
Recuperado/Resolvido com sequelas              4589
Name: count, dtype: int64

### GRAVIDADE

In [7]:
bronze.GRAVIDADE.value_counts()

GRAVIDADE
Outro efeito clinicamente significativo                                                                                                                                                                              153561
Hospitalização/Prolongamento de hospitalização                                                                                                                                                                        61677
Hospitalização/Prolongamento de hospitalização, Outro efeito clinicamente significativo                                                                                                                               16587
Resultou em óbito                                                                                                                                                                                                     12123
Ameaça à vida                                                                                                 

In [8]:
 
gravidades_unicas = (
    bronze['GRAVIDADE']
    .dropna()
    .astype(str)
    .str.split(r'\s*,\s*')   # separa pelos ","
    .explode()               # uma linha por item
    .str.strip()             # tira espaços extras
    .drop_duplicates()       # remove duplicados
    .sort_values()           # opcional: ordena
)

print(gravidades_unicas.to_list())


['Ameaça à vida', 'Anomalia congênita ou malformação ao nascer', 'Hospitalização/Prolongamento de hospitalização', 'Incapacidade persistente ou significativa', 'Outro efeito clinicamente significativo', 'Resultou em óbito']


In [9]:
for g in gravidades_unicas:
    print(g)

Ameaça à vida
Anomalia congênita ou malformação ao nascer
Hospitalização/Prolongamento de hospitalização
Incapacidade persistente ou significativa
Outro efeito clinicamente significativo
Resultou em óbito


### DURACAO

In [10]:
bronze.DURACAO.value_counts().head(10)

DURACAO
1 dia        30327
2 dia        10824
3 dia         7184
1 hora        7182
30 minuto     6654
4 dia         4573
5 dia         3983
0 dia         3611
2 hora        3427
20 minuto     3005
Name: count, dtype: int64

# 🥈 Silver

normalized data, medium quality


In [11]:
silver= bronze.copy()  
silver.head()

,IDENTIFICACAO_NOTIFICACAO,REACAO_EVTO_ADVERSO_MEDDRA_LLT,PT,HLT,HLGT,SOC,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO
0,BR-ANVISA-300000004,Coceira,Prurido,Prurido NCO,Quadros clínicos epidérmicos e dérmicos,Distúrbios dos tecidos cutâneos e subcutâneos,None,None,3 dia,Não,None,Recuperado/Resolvido,2026-01-15
1,BR-ANVISA-300000005,Edema periorbital,Edema periorbital,Distúrbios oculares NCO,Transtornos oculares NCO,Distúrbios oculares,20181122,20181122,None,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2026-01-15
2,BR-ANVISA-300000007,Exantema alérgico,Dermatite alérgica,Dermatite e eczema,Quadros clínicos epidérmicos e dérmicos,Distúrbios dos tecidos cutâneos e subcutâneos,20181115,None,2 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2026-01-15
3,BR-ANVISA-300000008,Flebite,Flebite,Flebite NCO,Infecções e inflamações vasculares,Distúrbios vasculares,20181025,None,5 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2026-01-15
4,BR-ANVISA-300000010,Parestesia,Parestesia,Parestesias e disestesias,Distúrbios neurológicos NCO,Distúrbios do sistema nervoso,201508,201508,None,Sim,Hospitalização/Prolongamento de hospitalização,Recuperado/Resolvido,2026-01-15


## ✅ hash

In [12]:
silver.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'REACAO_EVTO_ADVERSO_MEDDRA_LLT', 'PT',
       'HLT', 'HLGT', 'SOC', 'DATA_INICIO_HORA', 'DATA_FINAL_HORA', 'DURACAO',
       'GRAVE', 'GRAVIDADE', 'DESFECHO', 'ATUALIZADO'],
      dtype='object')

In [13]:
from utils import build_row_hash

hash_cols = ['IDENTIFICACAO_NOTIFICACAO', 'REACAO_EVTO_ADVERSO_MEDDRA_LLT',
       'PT', 'HLT', 'HLGT', 'SOC', 'DATA_INICIO_HORA', 'DATA_FINAL_HORA',
       'DURACAO', 'GRAVE', 'GRAVIDADE', 'DESFECHO']  # as colunas que devem entrar no hash
silver["HASH_BRONZE"] = build_row_hash(silver, hash_cols, algo="sha256", sep="|")

## ✅missing

In [14]:
silver.isna().sum()

IDENTIFICACAO_NOTIFICACAO              0
REACAO_EVTO_ADVERSO_MEDDRA_LLT     17292
PT                                 17292
HLT                                17292
HLGT                               17292
SOC                                17292
DATA_INICIO_HORA                  382098
DATA_FINAL_HORA                   619774
DURACAO                           732629
GRAVE                             182972
GRAVIDADE                         588461
DESFECHO                          142752
ATUALIZADO                             0
HASH_BRONZE                            0
dtype: int64

## ✅ chave_soc  chave_llt

In [15]:
from utils import meddra_match_pipeline

In [16]:
dim_soc = pd.read_parquet(Path(project_root) / "data/03_gold/dim_soc_llt/dim_soc_llt.parquet")
dim_soc.head()

,REACAO_CHAVE,PRIMARY_FLAG,SOC_CODE,SOC_NAME,SOC_ABBREV,PRIMARY_SOC,HLGT_CODE,HLGT_NAME,HLT_CODE,HLT_NAME,PT_CODE,PT_NAME,LLT_CODE,LLT_NAME
0,SOC_10005329_HLGT_10002086_HLT_10002042_PT_10002043_LLT_10002043,Y,10005329,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,10002086,Anemias não hemolíticas e depressão medular,10002042,Anemiais carenciais,10002043,Anemia por deficiência de folato,10002043,Anemia por deficiência de folato
1,SOC_10005329_HLGT_10002086_HLT_10002042_PT_10002043_LLT_10002044,Y,10005329,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,10002086,Anemias não hemolíticas e depressão medular,10002042,Anemiais carenciais,10002043,Anemia por deficiência de folato,10002044,Anemia por deficiência de ácido fólico
2,SOC_10005329_HLGT_10002086_HLT_10002042_PT_10002043_LLT_10002281,Y,10005329,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,10002086,Anemias não hemolíticas e depressão medular,10002042,Anemiais carenciais,10002043,Anemia por deficiência de folato,10002281,Anemia por carência de folato
3,SOC_10005329_HLGT_10002086_HLT_10002042_PT_10002043_LLT_10002282,Y,10005329,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,10002086,Anemias não hemolíticas e depressão medular,10002042,Anemiais carenciais,10002043,Anemia por deficiência de folato,10002282,Anemia por carência de ácido fólico
4,SOC_10005329_HLGT_10002086_HLT_10002042_PT_10002043_LLT_10016881,Y,10005329,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,10002086,Anemias não hemolíticas e depressão medular,10002042,Anemiais carenciais,10002043,Anemia por deficiência de folato,10016881,Anemia por carência de folato


In [17]:
dim_soc.columns

Index(['REACAO_CHAVE', 'PRIMARY_FLAG', 'SOC_CODE', 'SOC_NAME', 'SOC_ABBREV',
       'PRIMARY_SOC', 'HLGT_CODE', 'HLGT_NAME', 'HLT_CODE', 'HLT_NAME',
       'PT_CODE', 'PT_NAME', 'LLT_CODE', 'LLT_NAME'],
      dtype='object')

In [18]:
silver = silver.rename(columns={"REACAO_EVTO_ADVERSO_MEDDRA_LLT": "LLT"})
dim_silver = (
    silver[["SOC","HLGT","HLT", "PT","LLT"]]
    .value_counts()
    .reset_index(name='count')  # Converte para DataFrame
    .sort_values(by=["SOC","HLGT","HLT", "PT","LLT"])
)
dim_silver.head()

result = meddra_match_pipeline(
    dim_silver,
    dim_soc,
    fuzzy_threshold=60,
    output_mode='all'
) 

result = result.rename(columns={"SOC_VIGIMED":"SOC",
        "HLGT_VIGIMED":"HLGT",
        "HLT_VIGIMED":"HLT",
        "PT_VIGIMED":"PT",
        "LLT_VIGIMED":"LLT"}) 
result.head()

Match exato: 18454 registros
Sem match: 602 registros
Executando fuzzy matching com threshold=60...
Fuzzy matched: 602 registros
Sem match final: 0 registros


,SOC,HLGT,HLT,PT,LLT,count,LLT_NORM,PT_NORM,HLT_NORM,HLGT_NORM,SOC_NORM,REACAO_CHAVE,PRIMARY_FLAG,SOC_CODE,SOC_MEDDRA,SOC_ABBREV,PRIMARY_SOC,HLGT_CODE,HLGT_MEDDRA,HLT_CODE,HLT_MEDDRA,PT_CODE,PT_MEDDRA,LLT_CODE,LLT_MEDDRA,MATCH_TYPE,MATCH_SCORE
0,Circunstâncias sociais,Fatores relacionados ao gênero,Circunstâncias relacionadas à gravidez,Amamentação,Amamentação,9,amamentacao,amamentacao,circunstancias relacionadas a gravidez,fatores relacionados ao genero,circunstancias sociais,SOC_10041244_HLGT_10018057_HLT_10036569_PT_10006247_LLT_10006247,Y,10041244,Circunstâncias sociais,SocCi,10041244,10018057,Fatores relacionados ao gênero,10036569,Circunstâncias relacionadas à gravidez,10006247,Amamentação,10006247,Amamentação,EXACT,100.0
1,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Abstinência sexual,Abstinência sexual,3,abstinencia sexual,abstinencia sexual,questoes de sexualidade,fatores relacionados ao genero,circunstancias sociais,SOC_10041244_HLGT_10018057_HLT_10040489_PT_10086592_LLT_10086592,Y,10041244,Circunstâncias sociais,SocCi,10041244,10018057,Fatores relacionados ao gênero,10040489,Questões de sexualidade,10086592,Abstinência sexual,10086592,Abstinência sexual,EXACT,100.0
2,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Atividade sexual aumentada,Atividade sexual aumentada,1,atividade sexual aumentada,atividade sexual aumentada,questoes de sexualidade,fatores relacionados ao genero,circunstancias sociais,SOC_10041244_HLGT_10018057_HLT_10040489_PT_10055004_LLT_10055004,Y,10041244,Circunstâncias sociais,SocCi,10041244,10018057,Fatores relacionados ao gênero,10040489,Questões de sexualidade,10055004,Atividade sexual aumentada,10055004,Atividade sexual aumentada,EXACT,100.0
3,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Não consumação,Não consumação,6,nao consumacao,nao consumacao,questoes de sexualidade,fatores relacionados ao genero,circunstancias sociais,SOC_10041244_HLGT_10018057_HLT_10040489_PT_10029540_LLT_10029540,Y,10041244,Circunstâncias sociais,SocCi,10041244,10018057,Fatores relacionados ao gênero,10040489,Questões de sexualidade,10029540,Não consumação,10029540,Não consumação,EXACT,100.0
4,Circunstâncias sociais,Fatores relacionados à idade,Questões relacionadas à idade,Bebê,Neonato,1,neonato,bebe,questoes relacionadas a idade,fatores relacionados a idade,circunstancias sociais,SOC_10041244_HLGT_10001474_HLT_10001475_PT_10021731_LLT_10028977,Y,10041244,Circunstâncias sociais,SocCi,10041244,10001474,Fatores relacionados à idade,10001475,Questões relacionadas à idade,10021731,Bebê,10028977,Neonato,EXACT,100.0


In [19]:
silver.shape

(873682, 14)

In [20]:
silver.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'LLT', 'PT', 'HLT', 'HLGT', 'SOC',
       'DATA_INICIO_HORA', 'DATA_FINAL_HORA', 'DURACAO', 'GRAVE', 'GRAVIDADE',
       'DESFECHO', 'ATUALIZADO', 'HASH_BRONZE'],
      dtype='object')

In [21]:
hist_fat_reacoes = silver.merge(result[['REACAO_CHAVE','LLT', 'PT', 'HLT', 'HLGT','SOC']], on=['LLT', 'PT', 'HLT', 'HLGT','SOC'], how='left')
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,LLT,PT,HLT,HLGT,SOC,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO,HASH_BRONZE,REACAO_CHAVE
0,BR-ANVISA-300000004,Coceira,Prurido,Prurido NCO,Quadros clínicos epidérmicos e dérmicos,Distúrbios dos tecidos cutâneos e subcutâneos,None,None,3 dia,Não,None,Recuperado/Resolvido,2026-01-15,e0102770fa7fe4e3e3f58fe5a3675e13fffd3dc4721c0bdc390b6a0bb6bb8a59,SOC_10040785_HLGT_10014982_HLT_10049293_PT_10037087_LLT_10023084
1,BR-ANVISA-300000005,Edema periorbital,Edema periorbital,Distúrbios oculares NCO,Transtornos oculares NCO,Distúrbios oculares,20181122,20181122,None,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2026-01-15,21a5c47062a6ba174af4058e70e681ff97ba91db76067626020d5936dfd44c38,SOC_10015919_HLGT_10015917_HLT_10030032_PT_10034545_LLT_10054541
2,BR-ANVISA-300000007,Exantema alérgico,Dermatite alérgica,Dermatite e eczema,Quadros clínicos epidérmicos e dérmicos,Distúrbios dos tecidos cutâneos e subcutâneos,20181115,None,2 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2026-01-15,de3dc592331e18cf642659152efae63272db0a105f20bd6e69410b1a28ae8823,SOC_10040785_HLGT_10014982_HLT_10012435_PT_10012434_LLT_10057363
3,BR-ANVISA-300000008,Flebite,Flebite,Flebite NCO,Infecções e inflamações vasculares,Distúrbios vasculares,20181025,None,5 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2026-01-15,9d8d2f4fb5c70d57589cdf2ca4a6c5d7b73f62cb62403ed4fff2761725e8e890,SOC_10047065_HLGT_10082207_HLT_10052780_PT_10034879_LLT_10034879
4,BR-ANVISA-300000010,Parestesia,Parestesia,Parestesias e disestesias,Distúrbios neurológicos NCO,Distúrbios do sistema nervoso,201508,201508,None,Sim,Hospitalização/Prolongamento de hospitalização,Recuperado/Resolvido,2026-01-15,57864a0b52f197b8c908c0ae1fa00bdedfe7c44adab7049d2152d2be4ea39eb2,SOC_10029205_HLGT_10029305_HLT_10033788_PT_10033775_LLT_10033987


In [22]:
hist_fat_reacoes.shape

(873682, 15)

## ✅DATAS

In [23]:
col_dates = ["DATA_INICIO_HORA", "DATA_FINAL_HORA"]

for col in col_dates:
    if col in hist_fat_reacoes.columns:
        hist_fat_reacoes[col] = normalize_date_column(hist_fat_reacoes[col])

hist_fat_reacoes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 873682 entries, 0 to 873681
Data columns (total 15 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   IDENTIFICACAO_NOTIFICACAO  873682 non-null  object        
 1   LLT                        856390 non-null  object        
 2   PT                         856390 non-null  object        
 3   HLT                        856390 non-null  object        
 4   HLGT                       856390 non-null  object        
 5   SOC                        856390 non-null  object        
 6   DATA_INICIO_HORA           373466 non-null  datetime64[ns]
 7   DATA_FINAL_HORA            206890 non-null  datetime64[ns]
 8   DURACAO                    141053 non-null  object        
 9   GRAVE                      690710 non-null  object        
 10  GRAVIDADE                  285221 non-null  object        
 11  DESFECHO                   730930 non-null  object  

In [24]:
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,LLT,PT,HLT,HLGT,SOC,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO,HASH_BRONZE,REACAO_CHAVE
0,BR-ANVISA-300000004,Coceira,Prurido,Prurido NCO,Quadros clínicos epidérmicos e dérmicos,Distúrbios dos tecidos cutâneos e subcutâneos,NaT,NaT,3 dia,Não,None,Recuperado/Resolvido,2026-01-15,e0102770fa7fe4e3e3f58fe5a3675e13fffd3dc4721c0bdc390b6a0bb6bb8a59,SOC_10040785_HLGT_10014982_HLT_10049293_PT_10037087_LLT_10023084
1,BR-ANVISA-300000005,Edema periorbital,Edema periorbital,Distúrbios oculares NCO,Transtornos oculares NCO,Distúrbios oculares,2018-11-22,2018-11-22,None,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2026-01-15,21a5c47062a6ba174af4058e70e681ff97ba91db76067626020d5936dfd44c38,SOC_10015919_HLGT_10015917_HLT_10030032_PT_10034545_LLT_10054541
2,BR-ANVISA-300000007,Exantema alérgico,Dermatite alérgica,Dermatite e eczema,Quadros clínicos epidérmicos e dérmicos,Distúrbios dos tecidos cutâneos e subcutâneos,2018-11-15,NaT,2 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2026-01-15,de3dc592331e18cf642659152efae63272db0a105f20bd6e69410b1a28ae8823,SOC_10040785_HLGT_10014982_HLT_10012435_PT_10012434_LLT_10057363
3,BR-ANVISA-300000008,Flebite,Flebite,Flebite NCO,Infecções e inflamações vasculares,Distúrbios vasculares,2018-10-25,NaT,5 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2026-01-15,9d8d2f4fb5c70d57589cdf2ca4a6c5d7b73f62cb62403ed4fff2761725e8e890,SOC_10047065_HLGT_10082207_HLT_10052780_PT_10034879_LLT_10034879
4,BR-ANVISA-300000010,Parestesia,Parestesia,Parestesias e disestesias,Distúrbios neurológicos NCO,Distúrbios do sistema nervoso,NaT,NaT,None,Sim,Hospitalização/Prolongamento de hospitalização,Recuperado/Resolvido,2026-01-15,57864a0b52f197b8c908c0ae1fa00bdedfe7c44adab7049d2152d2be4ea39eb2,SOC_10029205_HLGT_10029305_HLT_10033788_PT_10033775_LLT_10033987


## ✅ GRAVE

In [25]:
hist_fat_reacoes.GRAVE.value_counts()

GRAVE
Não    405156
Sim    285554
Name: count, dtype: int64

In [26]:
 
grave_map = {
    "Desconhecido": 0,
    "Sim": 1,
    "Não": 2, 
}
hist_fat_reacoes['GRAVE_VALOR'] = hist_fat_reacoes['GRAVE'].fillna("Desconhecido") 
hist_fat_reacoes['GRAVE_CHAVE'] = hist_fat_reacoes['GRAVE'].map(grave_map).astype('Int64').fillna(0)
hist_fat_reacoes = hist_fat_reacoes.drop('GRAVE', axis=1)
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,LLT,PT,HLT,HLGT,SOC,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVIDADE,DESFECHO,ATUALIZADO,HASH_BRONZE,REACAO_CHAVE,GRAVE_VALOR,GRAVE_CHAVE
0,BR-ANVISA-300000004,Coceira,Prurido,Prurido NCO,Quadros clínicos epidérmicos e dérmicos,Distúrbios dos tecidos cutâneos e subcutâneos,NaT,NaT,3 dia,None,Recuperado/Resolvido,2026-01-15,e0102770fa7fe4e3e3f58fe5a3675e13fffd3dc4721c0bdc390b6a0bb6bb8a59,SOC_10040785_HLGT_10014982_HLT_10049293_PT_10037087_LLT_10023084,Não,2
1,BR-ANVISA-300000005,Edema periorbital,Edema periorbital,Distúrbios oculares NCO,Transtornos oculares NCO,Distúrbios oculares,2018-11-22,2018-11-22,None,Outro efeito clinicamente significativo,Recuperado/Resolvido,2026-01-15,21a5c47062a6ba174af4058e70e681ff97ba91db76067626020d5936dfd44c38,SOC_10015919_HLGT_10015917_HLT_10030032_PT_10034545_LLT_10054541,Sim,1
2,BR-ANVISA-300000007,Exantema alérgico,Dermatite alérgica,Dermatite e eczema,Quadros clínicos epidérmicos e dérmicos,Distúrbios dos tecidos cutâneos e subcutâneos,2018-11-15,NaT,2 dia,Outro efeito clinicamente significativo,Recuperado/Resolvido,2026-01-15,de3dc592331e18cf642659152efae63272db0a105f20bd6e69410b1a28ae8823,SOC_10040785_HLGT_10014982_HLT_10012435_PT_10012434_LLT_10057363,Sim,1
3,BR-ANVISA-300000008,Flebite,Flebite,Flebite NCO,Infecções e inflamações vasculares,Distúrbios vasculares,2018-10-25,NaT,5 dia,Outro efeito clinicamente significativo,Recuperado/Resolvido,2026-01-15,9d8d2f4fb5c70d57589cdf2ca4a6c5d7b73f62cb62403ed4fff2761725e8e890,SOC_10047065_HLGT_10082207_HLT_10052780_PT_10034879_LLT_10034879,Sim,1
4,BR-ANVISA-300000010,Parestesia,Parestesia,Parestesias e disestesias,Distúrbios neurológicos NCO,Distúrbios do sistema nervoso,NaT,NaT,None,Hospitalização/Prolongamento de hospitalização,Recuperado/Resolvido,2026-01-15,57864a0b52f197b8c908c0ae1fa00bdedfe7c44adab7049d2152d2be4ea39eb2,SOC_10029205_HLGT_10029305_HLT_10033788_PT_10033775_LLT_10033987,Sim,1


In [27]:
hist_fat_reacoes[['GRAVE_CHAVE', 'GRAVE_VALOR']].drop_duplicates()

,GRAVE_CHAVE,GRAVE_VALOR
0,2,Não
1,1,Sim
36,0,Desconhecido


## ✅ DESFECHO

In [28]:
hist_fat_reacoes.DESFECHO.value_counts()

DESFECHO
Recuperado/Resolvido                         303064
Desconhecido                                 240842
Não Recuperado/Não Resolvido/Em andamento     90692
Em recuperação/Resolvendo                     72126
Fatal/Óbito                                   19617
Recuperado/Resolvido com sequelas              4589
Name: count, dtype: int64

In [29]:
desfecho_map = {
    "Desconhecido": 0, 
    "Não Recuperado/Não Resolvido/Em andamento": 1, 
    "Em recuperação/Resolvendo": 2, 
    "Recuperado/Resolvido": 3,
    "Fatal/Óbito": 4, 
    "Recuperado/Resolvido com sequelas": 5, 
}
hist_fat_reacoes['DESFECHO_VALOR'] = hist_fat_reacoes['DESFECHO'].fillna("Desconhecido")  
hist_fat_reacoes['DESFECHO_CHAVE'] = hist_fat_reacoes['DESFECHO'].map(desfecho_map).astype('Int64').fillna(0)
hist_fat_reacoes = hist_fat_reacoes.drop('DESFECHO', axis=1)
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,LLT,PT,HLT,HLGT,SOC,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVIDADE,ATUALIZADO,HASH_BRONZE,REACAO_CHAVE,GRAVE_VALOR,GRAVE_CHAVE,DESFECHO_VALOR,DESFECHO_CHAVE
0,BR-ANVISA-300000004,Coceira,Prurido,Prurido NCO,Quadros clínicos epidérmicos e dérmicos,Distúrbios dos tecidos cutâneos e subcutâneos,NaT,NaT,3 dia,None,2026-01-15,e0102770fa7fe4e3e3f58fe5a3675e13fffd3dc4721c0bdc390b6a0bb6bb8a59,SOC_10040785_HLGT_10014982_HLT_10049293_PT_10037087_LLT_10023084,Não,2,Recuperado/Resolvido,3
1,BR-ANVISA-300000005,Edema periorbital,Edema periorbital,Distúrbios oculares NCO,Transtornos oculares NCO,Distúrbios oculares,2018-11-22,2018-11-22,None,Outro efeito clinicamente significativo,2026-01-15,21a5c47062a6ba174af4058e70e681ff97ba91db76067626020d5936dfd44c38,SOC_10015919_HLGT_10015917_HLT_10030032_PT_10034545_LLT_10054541,Sim,1,Recuperado/Resolvido,3
2,BR-ANVISA-300000007,Exantema alérgico,Dermatite alérgica,Dermatite e eczema,Quadros clínicos epidérmicos e dérmicos,Distúrbios dos tecidos cutâneos e subcutâneos,2018-11-15,NaT,2 dia,Outro efeito clinicamente significativo,2026-01-15,de3dc592331e18cf642659152efae63272db0a105f20bd6e69410b1a28ae8823,SOC_10040785_HLGT_10014982_HLT_10012435_PT_10012434_LLT_10057363,Sim,1,Recuperado/Resolvido,3
3,BR-ANVISA-300000008,Flebite,Flebite,Flebite NCO,Infecções e inflamações vasculares,Distúrbios vasculares,2018-10-25,NaT,5 dia,Outro efeito clinicamente significativo,2026-01-15,9d8d2f4fb5c70d57589cdf2ca4a6c5d7b73f62cb62403ed4fff2761725e8e890,SOC_10047065_HLGT_10082207_HLT_10052780_PT_10034879_LLT_10034879,Sim,1,Recuperado/Resolvido,3
4,BR-ANVISA-300000010,Parestesia,Parestesia,Parestesias e disestesias,Distúrbios neurológicos NCO,Distúrbios do sistema nervoso,NaT,NaT,None,Hospitalização/Prolongamento de hospitalização,2026-01-15,57864a0b52f197b8c908c0ae1fa00bdedfe7c44adab7049d2152d2be4ea39eb2,SOC_10029205_HLGT_10029305_HLT_10033788_PT_10033775_LLT_10033987,Sim,1,Recuperado/Resolvido,3


In [30]:
hist_fat_reacoes[['DESFECHO_CHAVE', 'DESFECHO_VALOR']].drop_duplicates()

,DESFECHO_CHAVE,DESFECHO_VALOR
0,3,Recuperado/Resolvido
5,0,Desconhecido
39,2,Em recuperação/Resolvendo
53,1,Não Recuperado/Não Resolvido/Em andamento
64,4,Fatal/Óbito
331,5,Recuperado/Resolvido com sequelas


## ✅ DURACAO

In [31]:
from utils import normalize_duracao

In [32]:
hist_fat_reacoes.DURACAO.value_counts().head(10)

DURACAO
1 dia        30327
2 dia        10824
3 dia         7184
1 hora        7182
30 minuto     6654
4 dia         4573
5 dia         3983
0 dia         3611
2 hora        3427
20 minuto     3005
Name: count, dtype: int64

In [33]:
hist_fat_reacoes = normalize_duracao(hist_fat_reacoes, coluna="DURACAO")

In [34]:
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,LLT,PT,HLT,HLGT,SOC,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVIDADE,ATUALIZADO,HASH_BRONZE,REACAO_CHAVE,GRAVE_VALOR,GRAVE_CHAVE,DESFECHO_VALOR,DESFECHO_CHAVE,DURACAO_TIPO_VALOR,DURACAO_TIPO_CHAVE,DURACAO_VALOR
0,BR-ANVISA-300000004,Coceira,Prurido,Prurido NCO,Quadros clínicos epidérmicos e dérmicos,Distúrbios dos tecidos cutâneos e subcutâneos,NaT,NaT,3 dia,None,2026-01-15,e0102770fa7fe4e3e3f58fe5a3675e13fffd3dc4721c0bdc390b6a0bb6bb8a59,SOC_10040785_HLGT_10014982_HLT_10049293_PT_10037087_LLT_10023084,Não,2,Recuperado/Resolvido,3,DIA,4,3.0
1,BR-ANVISA-300000005,Edema periorbital,Edema periorbital,Distúrbios oculares NCO,Transtornos oculares NCO,Distúrbios oculares,2018-11-22,2018-11-22,None,Outro efeito clinicamente significativo,2026-01-15,21a5c47062a6ba174af4058e70e681ff97ba91db76067626020d5936dfd44c38,SOC_10015919_HLGT_10015917_HLT_10030032_PT_10034545_LLT_10054541,Sim,1,Recuperado/Resolvido,3,DESCONHECIDO,0,NaN
2,BR-ANVISA-300000007,Exantema alérgico,Dermatite alérgica,Dermatite e eczema,Quadros clínicos epidérmicos e dérmicos,Distúrbios dos tecidos cutâneos e subcutâneos,2018-11-15,NaT,2 dia,Outro efeito clinicamente significativo,2026-01-15,de3dc592331e18cf642659152efae63272db0a105f20bd6e69410b1a28ae8823,SOC_10040785_HLGT_10014982_HLT_10012435_PT_10012434_LLT_10057363,Sim,1,Recuperado/Resolvido,3,DIA,4,2.0
3,BR-ANVISA-300000008,Flebite,Flebite,Flebite NCO,Infecções e inflamações vasculares,Distúrbios vasculares,2018-10-25,NaT,5 dia,Outro efeito clinicamente significativo,2026-01-15,9d8d2f4fb5c70d57589cdf2ca4a6c5d7b73f62cb62403ed4fff2761725e8e890,SOC_10047065_HLGT_10082207_HLT_10052780_PT_10034879_LLT_10034879,Sim,1,Recuperado/Resolvido,3,DIA,4,5.0
4,BR-ANVISA-300000010,Parestesia,Parestesia,Parestesias e disestesias,Distúrbios neurológicos NCO,Distúrbios do sistema nervoso,NaT,NaT,None,Hospitalização/Prolongamento de hospitalização,2026-01-15,57864a0b52f197b8c908c0ae1fa00bdedfe7c44adab7049d2152d2be4ea39eb2,SOC_10029205_HLGT_10029305_HLT_10033788_PT_10033775_LLT_10033987,Sim,1,Recuperado/Resolvido,3,DESCONHECIDO,0,NaN


In [35]:
hist_fat_reacoes.DURACAO_VALOR.value_counts().head(15)

DURACAO_VALOR
1.0     41538
2.0     16803
3.0     10764
30.0     7287
5.0      6761
4.0      6665
10.0     4691
6.0      3934
0.0      3754
20.0     3728
7.0      3587
15.0     3119
8.0      2904
40.0     2234
12.0     1761
Name: count, dtype: int64

### GRAVIDADE

In [36]:
from utils import expandir_gravidade_wide

In [37]:
hist_fat_reacoes = expandir_gravidade_wide(hist_fat_reacoes, col='GRAVIDADE', prefix='GRAVIDADE_')
#hist_fat_reacoes = hist_fat_reacoes.drop('GRAVIDADE', axis=1)
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,LLT,PT,HLT,HLGT,SOC,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVIDADE,ATUALIZADO,HASH_BRONZE,REACAO_CHAVE,GRAVE_VALOR,GRAVE_CHAVE,DESFECHO_VALOR,DESFECHO_CHAVE,DURACAO_TIPO_VALOR,DURACAO_TIPO_CHAVE,DURACAO_VALOR,GRAVIDADE_RESULTADO_OBITO,GRAVIDADE_AMEACA_VIDA,GRAVIDADE_INCAPACIDADE,GRAVIDADE_HOSPITALIZACAO,GRAVIDADE_OUTRO_EFEITO,GRAVIDADE_ANOMALIA_CONGENITA
0,BR-ANVISA-300000004,Coceira,Prurido,Prurido NCO,Quadros clínicos epidérmicos e dérmicos,Distúrbios dos tecidos cutâneos e subcutâneos,NaT,NaT,3 dia,None,2026-01-15,e0102770fa7fe4e3e3f58fe5a3675e13fffd3dc4721c0bdc390b6a0bb6bb8a59,SOC_10040785_HLGT_10014982_HLT_10049293_PT_10037087_LLT_10023084,Não,2,Recuperado/Resolvido,3,DIA,4,3.0,0,0,0,0,0,0
1,BR-ANVISA-300000005,Edema periorbital,Edema periorbital,Distúrbios oculares NCO,Transtornos oculares NCO,Distúrbios oculares,2018-11-22,2018-11-22,None,Outro efeito clinicamente significativo,2026-01-15,21a5c47062a6ba174af4058e70e681ff97ba91db76067626020d5936dfd44c38,SOC_10015919_HLGT_10015917_HLT_10030032_PT_10034545_LLT_10054541,Sim,1,Recuperado/Resolvido,3,DESCONHECIDO,0,NaN,0,0,0,0,1,0
2,BR-ANVISA-300000007,Exantema alérgico,Dermatite alérgica,Dermatite e eczema,Quadros clínicos epidérmicos e dérmicos,Distúrbios dos tecidos cutâneos e subcutâneos,2018-11-15,NaT,2 dia,Outro efeito clinicamente significativo,2026-01-15,de3dc592331e18cf642659152efae63272db0a105f20bd6e69410b1a28ae8823,SOC_10040785_HLGT_10014982_HLT_10012435_PT_10012434_LLT_10057363,Sim,1,Recuperado/Resolvido,3,DIA,4,2.0,0,0,0,0,1,0
3,BR-ANVISA-300000008,Flebite,Flebite,Flebite NCO,Infecções e inflamações vasculares,Distúrbios vasculares,2018-10-25,NaT,5 dia,Outro efeito clinicamente significativo,2026-01-15,9d8d2f4fb5c70d57589cdf2ca4a6c5d7b73f62cb62403ed4fff2761725e8e890,SOC_10047065_HLGT_10082207_HLT_10052780_PT_10034879_LLT_10034879,Sim,1,Recuperado/Resolvido,3,DIA,4,5.0,0,0,0,0,1,0
4,BR-ANVISA-300000010,Parestesia,Parestesia,Parestesias e disestesias,Distúrbios neurológicos NCO,Distúrbios do sistema nervoso,NaT,NaT,None,Hospitalização/Prolongamento de hospitalização,2026-01-15,57864a0b52f197b8c908c0ae1fa00bdedfe7c44adab7049d2152d2be4ea39eb2,SOC_10029205_HLGT_10029305_HLT_10033788_PT_10033775_LLT_10033987,Sim,1,Recuperado/Resolvido,3,DESCONHECIDO,0,NaN,0,0,0,1,0,0


### hash

In [38]:
hist_fat_reacoes.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'LLT', 'PT', 'HLT', 'HLGT', 'SOC',
       'DATA_INICIO_HORA', 'DATA_FINAL_HORA', 'DURACAO', 'GRAVIDADE',
       'ATUALIZADO', 'HASH_BRONZE', 'REACAO_CHAVE', 'GRAVE_VALOR',
       'GRAVE_CHAVE', 'DESFECHO_VALOR', 'DESFECHO_CHAVE', 'DURACAO_TIPO_VALOR',
       'DURACAO_TIPO_CHAVE', 'DURACAO_VALOR', 'GRAVIDADE_RESULTADO_OBITO',
       'GRAVIDADE_AMEACA_VIDA', 'GRAVIDADE_INCAPACIDADE',
       'GRAVIDADE_HOSPITALIZACAO', 'GRAVIDADE_OUTRO_EFEITO',
       'GRAVIDADE_ANOMALIA_CONGENITA'],
      dtype='object')

In [39]:
hist_fat_reacoes["HASH_SILVER"] = build_row_hash(
    hist_fat_reacoes, 
    hist_fat_reacoes.columns.difference(['ATUALIZADO', 'HASH_BRONZE']).tolist(), 
    algo="sha256", 
    sep="|"
)

In [40]:
cols = ['IDENTIFICACAO_NOTIFICACAO', 'REACAO_CHAVE','LLT', 'PT', 'HLT', 'HLGT', 'SOC',
       'DATA_INICIO_HORA', 'DATA_FINAL_HORA', 'DURACAO', 'GRAVIDADE',
       'ATUALIZADO', 'HASH_BRONZE', 'GRAVE_VALOR',
       'GRAVE_CHAVE', 'DESFECHO_VALOR', 'DESFECHO_CHAVE', 'DURACAO_TIPO_VALOR',
       'DURACAO_TIPO_CHAVE', 'DURACAO_VALOR', 'GRAVIDADE_RESULTADO_OBITO',
       'GRAVIDADE_AMEACA_VIDA', 'GRAVIDADE_INCAPACIDADE',
       'GRAVIDADE_HOSPITALIZACAO', 'GRAVIDADE_OUTRO_EFEITO',
       'GRAVIDADE_ANOMALIA_CONGENITA']

In [41]:
hist_fat_reacoes[cols].to_parquet(Path(project_root) / "data/02_silver/hist_fat_reacoes/hist_fat_reacoes.parquet", index=False)

# 🥇 Gold


In [42]:
hist_fat_reacoes.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'LLT', 'PT', 'HLT', 'HLGT', 'SOC',
       'DATA_INICIO_HORA', 'DATA_FINAL_HORA', 'DURACAO', 'GRAVIDADE',
       'ATUALIZADO', 'HASH_BRONZE', 'REACAO_CHAVE', 'GRAVE_VALOR',
       'GRAVE_CHAVE', 'DESFECHO_VALOR', 'DESFECHO_CHAVE', 'DURACAO_TIPO_VALOR',
       'DURACAO_TIPO_CHAVE', 'DURACAO_VALOR', 'GRAVIDADE_RESULTADO_OBITO',
       'GRAVIDADE_AMEACA_VIDA', 'GRAVIDADE_INCAPACIDADE',
       'GRAVIDADE_HOSPITALIZACAO', 'GRAVIDADE_OUTRO_EFEITO',
       'GRAVIDADE_ANOMALIA_CONGENITA', 'HASH_SILVER'],
      dtype='object')

In [43]:
gold_cols = ['IDENTIFICACAO_NOTIFICACAO',
        'DATA_INICIO_HORA', 
        'DATA_FINAL_HORA', 
       'REACAO_CHAVE', 
       'GRAVE_CHAVE', 
       'GRAVE_VALOR', 
       'DESFECHO_CHAVE', 
       'DESFECHO_VALOR',
       'DURACAO_TIPO_CHAVE', 
       'DURACAO_TIPO_VALOR',
       'DURACAO_VALOR', 
       'GRAVIDADE_RESULTADO_OBITO', 
       'GRAVIDADE_AMEACA_VIDA',
       'GRAVIDADE_INCAPACIDADE', 
       'GRAVIDADE_HOSPITALIZACAO',
       'GRAVIDADE_OUTRO_EFEITO', 
       'GRAVIDADE_ANOMALIA_CONGENITA',
       
       'ATUALIZADO', 
       'HASH_BRONZE', 
       'HASH_SILVER']

In [44]:
fat = hist_fat_reacoes[gold_cols].copy()
fat["HASH_GOLD"] = build_row_hash(fat, gold_cols, algo="sha256", sep="|")

In [45]:

fat.to_parquet(Path(project_root) / "data/03_gold/fat_reacoes/fat_reacoes.parquet", index=False)

In [46]:
fat.head()

,IDENTIFICACAO_NOTIFICACAO,DATA_INICIO_HORA,DATA_FINAL_HORA,REACAO_CHAVE,GRAVE_CHAVE,GRAVE_VALOR,DESFECHO_CHAVE,DESFECHO_VALOR,DURACAO_TIPO_CHAVE,DURACAO_TIPO_VALOR,DURACAO_VALOR,GRAVIDADE_RESULTADO_OBITO,GRAVIDADE_AMEACA_VIDA,GRAVIDADE_INCAPACIDADE,GRAVIDADE_HOSPITALIZACAO,GRAVIDADE_OUTRO_EFEITO,GRAVIDADE_ANOMALIA_CONGENITA,ATUALIZADO,HASH_BRONZE,HASH_SILVER,HASH_GOLD
0,BR-ANVISA-300000004,NaT,NaT,SOC_10040785_HLGT_10014982_HLT_10049293_PT_10037087_LLT_10023084,2,Não,3,Recuperado/Resolvido,4,DIA,3.0,0,0,0,0,0,0,2026-01-15,e0102770fa7fe4e3e3f58fe5a3675e13fffd3dc4721c0bdc390b6a0bb6bb8a59,8c2bd3985ba5d5c3c38c928ced3769124a9f33ef2ff3f15bedbf239548c3bcb8,31c44de9604bad343edd29f25f1317be500fc6836be77b730a65cc9cdf7d1b3a
1,BR-ANVISA-300000005,2018-11-22,2018-11-22,SOC_10015919_HLGT_10015917_HLT_10030032_PT_10034545_LLT_10054541,1,Sim,3,Recuperado/Resolvido,0,DESCONHECIDO,NaN,0,0,0,0,1,0,2026-01-15,21a5c47062a6ba174af4058e70e681ff97ba91db76067626020d5936dfd44c38,056c972fd2c77ca0bfdae5b5950379182d491a356c52dc7fe5c961d14b250d73,a270a06c9ee9c3fc99b9f5b794a28c5768943f93594ab712cb0a895593cae751
2,BR-ANVISA-300000007,2018-11-15,NaT,SOC_10040785_HLGT_10014982_HLT_10012435_PT_10012434_LLT_10057363,1,Sim,3,Recuperado/Resolvido,4,DIA,2.0,0,0,0,0,1,0,2026-01-15,de3dc592331e18cf642659152efae63272db0a105f20bd6e69410b1a28ae8823,d4e262d34db9025a192d03c646e5e80e9c8fb0c4aa1502e7d4ce3088bf33e660,a76edc9d86503bc7d4bdfcebd76a32d00c3b7d3a33da1ec93b493ccd16c963a9
3,BR-ANVISA-300000008,2018-10-25,NaT,SOC_10047065_HLGT_10082207_HLT_10052780_PT_10034879_LLT_10034879,1,Sim,3,Recuperado/Resolvido,4,DIA,5.0,0,0,0,0,1,0,2026-01-15,9d8d2f4fb5c70d57589cdf2ca4a6c5d7b73f62cb62403ed4fff2761725e8e890,4eabc70a995135a787ee6752809e02515fb883b7a440ca5de39fbe5e9489ff23,ff8b7ac0278345f7c82717d8c98584812d3aaa65a0874357f87d5cc33d4bb981
4,BR-ANVISA-300000010,NaT,NaT,SOC_10029205_HLGT_10029305_HLT_10033788_PT_10033775_LLT_10033987,1,Sim,3,Recuperado/Resolvido,0,DESCONHECIDO,NaN,0,0,0,1,0,0,2026-01-15,57864a0b52f197b8c908c0ae1fa00bdedfe7c44adab7049d2152d2be4ea39eb2,cd8fd071e7ae0d82b9cd7072788f78370e54beec1789b4e03d0cf4daf34a1dd3,4b225a64ebed2801d542c6c51f3948d6e36bd4c70d49105acbace05b71dcb877
